# FSR v2 Preprocessor Standalone Runner (Databricks)

This notebook runs `common/fsr_v2/preprocessor_v2.py` in isolation for a single PDF.

What it does:
- Parses the PDF using the same parser path as P1 (`pymupdf_v1.0` via `parse_pymupdf`)
- Builds preprocessor `ctx` explicitly
- Runs `preprocessor.preprocess(ctx)` without LLM/enrichment/table writes
- Prints metadata/hints/regions
- Verifies region attribution around a target ESN token (selected per bug case)
- Saves raw JSON output as a debug artifact

This version is pre-configured for bug 3, 4, and 6 section cases.
Select one entry by `selected_case_index` in the config cell.

Expected runtime: usually under a few minutes (depends on cluster warmup + PDF size).

In [1]:
# Databricks dependency (safe to re-run)
%pip install -q pymupdf


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Databricks runtime restart after %pip install
# Uncomment in Databricks if needed:
# dbutils.library.restartPython()

In [ ]:
import json
import os
import sys
from datetime import datetime, timezone
from pathlib import Path
from types import SimpleNamespace


OUTPUT_DIR = "/Workspace/Users/madhurima.saxena@gevernova.com/fsr-preprocessor-v2-output-standalone"
print("Using OUTPUT_DIR:", OUTPUT_DIR)


def _find_repo_root(start: Path, target_name: str = "pw_sdg_ai_ser_repo") -> Path:
    p = start.resolve()
    while True:
        if p.name == target_name:
            return p
        if p.parent == p:
            raise RuntimeError(f"Could not find repo root '{target_name}' from {start}")
        p = p.parent


repo_root = _find_repo_root(Path.cwd())
silver_etl = repo_root / "silver" / "src" / "etl"

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if str(silver_etl) not in sys.path:
    sys.path.insert(0, str(silver_etl))

print("repo_root:", repo_root)
print("silver_etl:", silver_etl)

repo_root: /home/u560060992/dbx/pw_sdg_ai_ser_repo
silver_etl: /home/u560060992/dbx/pw_sdg_ai_ser_repo/silver/src/etl


In [ ]:
import importlib
from fsr_v2.parsing import parse_pymupdf
from fsr_v2.input import _lookup_target_docs
from common.fsr_v2 import preprocessor_v2 as preprocessor

# Force reload so Databricks reruns pick up synced preprocessor changes
preprocessor = importlib.reload(preprocessor)


def _resolve_volume_path(document_id_raw: str, document_id: str, volume_roots: list[str]):
    """Resolve validation files through the production target-discovery helper."""
    if "dbutils" not in globals():
        raise RuntimeError("Databricks dbutils is required for pipeline-backed volume discovery.")

    matches = _lookup_target_docs(
        volume_paths=volume_roots,
        target_document_ids=[document_id_raw],
        existing_doc_ids=set(),
        dbutils_client=dbutils,
    )
    if not matches:
        return f"{volume_roots[0].rstrip('/')}/{document_id}", volume_roots[0], False

    match = matches[0]
    volume_path = match["volume_path"]
    selected_root = next(
        (root for root in volume_roots if volume_path.startswith(f"{root.rstrip('/')}/")),
        volume_roots[0],
    )
    return volume_path, selected_root, True


# Bug 3/4/6 focused cases from the reviewed bug list.
test_cases = [
    {
        "name": "Bug 3 - section header without number",
        "bug_id": 3,
        "document_id": "35803273-2440-4d36-87fe-e45f7f0e5467_605011422-40815-270t483-final_master_report",
        "target_esn_for_region_check": None,
        "expected_esns": [],
    },
    {
        "name": "Bug 3 - section header without number",
        "bug_id": 3,
        "document_id": "806975e9-062b-4da3-9ae7-5881a5daffd1_204049598-39387-297652-final_master_report",
        "target_esn_for_region_check": None,
        "expected_esns": [],
    },
    {
        "name": "Bug 3 - missing/misaligned section boundaries",
        "bug_id": 3,
        "document_id": "5b688732-39f2-48d2-a887-3239f258d28b",
        "target_esn_for_region_check": None,
        "expected_esns": [],
    },
    {
        "name": "Bug 4 - subsection fallback and multi-ESN",
        "bug_id": 4,
        "document_id": "32689520-afed-4dd4-a895-20afed7dd4d2",
        "target_esn_for_region_check": None,
        "expected_esns": [],
    },
    {
        "name": "Bug 4 - two GT ESN issue",
        "bug_id": 4,
        "document_id": "b25c94da-d954-4283-9c94-dad954a28307",
        "target_esn_for_region_check": None,
        "expected_esns": [],
    },
    {
        "name": "Bug 4 - two GT and two GEN ESN issue",
        "bug_id": 4,
        "document_id": "b1cdbc80-364f-4240-8dbc-80364f1240fa",
        "target_esn_for_region_check": None,
        "expected_esns": [],
    },
    {
        "name": "Bug 4 - subsection should flip back to turbine",
        "bug_id": 4,
        "document_id": "fcb1511e-596a-4a56-b151-1e596afa569c",
        "target_esn_for_region_check": None,
        "expected_esns": [],
    },
    {
        "name": "Bug 4 - subsection should flip back to turbine",
        "bug_id": 4,
        "document_id": "af693a98-1e5c-499d-aa10-cccc54885c64",
        "target_esn_for_region_check": "337X766",
        "expected_esns": ["337X766", "297652"],
    },
    {
        "name": "Bug 6 - wrong Generator ESN assignment",
        "bug_id": 6,
        "document_id": "b775cf29-8b42-4a83-af21-53075fef0802",
        "target_esn_for_region_check": "337X765",
        "expected_esns": ["337X765", "297651"],
    },
    {
        "name": "Bug 6 - generator context without explicit generator ESN",
        "bug_id": 6,
        "document_id": "b7b347fd-0b59-4c67-b609-9ad2c35105cc",
        "target_esn_for_region_check": None,
        "expected_esns": [],
    },
]

# Set this to switch the document under test.
selected_case_index = 0
selected_case = test_cases[selected_case_index]

document_id_raw = selected_case["document_id"]
document_id = document_id_raw.split("_", 1)[0].rstrip(".")

# Scan across both mounted roots, first hit wins.
volume_roots = [
    "/Volumes/viud/ing_ud_fieldvision/fv_field_service_report",
    "/Volumes/vaid/ai_std_con_field_service_report/fsr_v2_test",
]
volume_path, selected_volume_root, file_in_any_volume = _resolve_volume_path(
    document_id_raw, document_id, volume_roots
)

target_esn_for_region_check = selected_case["target_esn_for_region_check"]
expected_esns_for_debug = selected_case["expected_esns"]

# Preprocessor only emits fields explicitly requested here.
# Add all_esns/st_esn for better debug visibility.
fields = [
    {"name": "primary_esn"},
    {"name": "primary_equip_type"},
    {"name": "primary_technology_code"},
    {"name": "inactive_esns"},
    {"name": "gt_esn"},
    {"name": "gen_esn"},
    {"name": "st_esn"},
    {"name": "all_esns"},
    {"name": "outage_start_date"},
    {"name": "outage_end_date"},
    {"name": "report_issued_date"},
    {"name": "document_name"},
]

print("Selected:", selected_case["name"])
print("Configured document raw:", document_id_raw)
print("Configured document normalized:", document_id)
print("Volume roots scanned:", volume_roots)
print("Selected volume root:", selected_volume_root)
print("Resolved volume path:", volume_path)
print("Document found through pipeline discovery:", file_in_any_volume)
print("Expected ESNs in doc context:", expected_esns_for_debug)

Selected: Bug 3 - section header without number
Configured document raw: 35803273-2440-4d36-87fe-e45f7f0e5467_605011422-40815-270t483-final_master_report
Configured document normalized: 35803273-2440-4d36-87fe-e45f7f0e5467
Volume roots scanned: ['/Volumes/viud/ing_ud_fieldvision/fv_field_service_report', '/Volumes/vaid/ai_std_con_field_service_report/fsr_v2_test']
Selected volume root: /Volumes/viud/ing_ud_fieldvision/fv_field_service_report
Resolved volume path: /Volumes/viud/ing_ud_fieldvision/fv_field_service_report/35803273-2440-4d36-87fe-e45f7f0e5467
Document found in scanned roots: False
Expected ESNs in doc context: []


In [ ]:
doc = {
    "document_id": document_id,
    "volume_path": volume_path,
}

file_in_volume = os.path.exists(volume_path)

if not file_in_volume:
    print("SKIP: file is not present in volume path:")
    print(" ", volume_path)
    parsed = SimpleNamespace(
        pages=[],
        page_offsets=[],
        full_text="",
        filename=document_id,
    )
else:
    parsed = parse_pymupdf(doc)

print("Parse stage complete")
print("  file_in_volume:", file_in_volume)
print("  pages:", len(parsed.pages))
print("  chars:", len(parsed.full_text))
print("  first page offset:", parsed.page_offsets[0] if parsed.page_offsets else None)
print("  filename:", parsed.filename)

In [ ]:
ctx = SimpleNamespace(
    pages=parsed.pages,
    page_offsets=parsed.page_offsets,
    full_text=parsed.full_text,
    fields=fields,
    filename=parsed.filename,
)

# Wire IBAT resolver so subsection ESNs (e.g. Generator inside a GT section) are resolved.
IBAT_TABLE = "vgpd.prm_std_views.ibat_equipment_mst"
ibat_resolver = None
try:
    from fsr_v2.metadata_processor_v2 import _make_ibat_resolver
    ibat_resolver = _make_ibat_resolver(spark, IBAT_TABLE)
    print("IBAT resolver enabled:", IBAT_TABLE)
except (NameError, ImportError) as _e:
    print("IBAT resolver disabled (no Spark / import unavailable):", _e)

result = preprocessor.preprocess(ctx, ibat_resolver=ibat_resolver)

metadata_raw = result.get("metadata", {}) or {}
regions = result.get("regions", []) or []
hints = result.get("hints", "") or ""

# Emit a stable, complete metadata shape even when some fields are missing.
expected_metadata_fields = [
    "document_name",
    "primary_esn",
    "primary_equip_type",
    "primary_technology_code",
    "gt_esn",
    "gen_esn",
    "st_esn",
    "all_esns",
    "inactive_esns",
    "outage_start_date",
    "outage_end_date",
    "report_issued_date",
]

metadata = {k: metadata_raw.get(k, "") for k in expected_metadata_fields}
# Preserve any extra keys preprocessor may emit in future.
for k, v in metadata_raw.items():
    if k not in metadata:
        metadata[k] = v

inactive_esns_raw = metadata.get("inactive_esns", "")
inactive_esns = [e.strip() for e in inactive_esns_raw.split(",") if e.strip()] if inactive_esns_raw else []

In [ ]:
# Region-level check around target ESN mention
needle = target_esn_for_region_check

if not needle:
    print("No target_esn_for_region_check configured for this case. Skipping region token check cleanly.")
else:
    idx = parsed.full_text.find(needle)
    print("target token:", needle)
    print("char index:", idx)

    if idx >= 0:
        hit_regions = [
            r for r in regions
            if idx >= int(r.get("start", -1)) and idx <= int(r.get("end", -1))
        ]
        print("covering regions:", len(hit_regions))
        if hit_regions:
            print(json.dumps(hit_regions, indent=2))
        else:
            print("No region covers this index. Possible coverage gap.")
    else:
        print("Target token not found in parsed.full_text. Check parser output/text normalization.")

In [ ]:
# Optional quick text snippets for manual review around ESNs expected in the selected case
tokens_to_review = [t for t in expected_esns_for_debug + [target_esn_for_region_check] if t]
tokens_to_review = list(dict.fromkeys(tokens_to_review))
for token in tokens_to_review:
    pos = parsed.full_text.find(token)
    print("\n---", token, "at", pos, "---")
    if pos >= 0:
        lo = max(0, pos - 250)
        hi = min(len(parsed.full_text), pos + 250)
        print(parsed.full_text[lo:hi])
    else:
        print("Token not found")

In [ ]:
artifact = {
    "document_id": document_id,
    "volume_path": volume_path,
    "run_utc": datetime.now(timezone.utc).isoformat(),
    "page_count": len(parsed.pages),
    "char_count": len(parsed.full_text),
    "metadata": metadata,
    "hints": hints,
    "regions": regions,
    "inactive_esns_emitted": inactive_esns,
}

artifact_json = json.dumps(artifact, indent=2)
print(artifact_json)

import os as _os
_os.makedirs(OUTPUT_DIR, exist_ok=True)
out_path = f"{OUTPUT_DIR}/{document_id}.json"
with open(out_path, "w") as f:
    f.write(artifact_json)
print(f"\nSaved: {out_path}")

In [ ]:
# Batch runner compatibility: route both invocation styles through production discovery.
def _resolve_volume_path(document_id_raw, document_id_or_roots, volume_roots=None):
    if volume_roots is None:
        document_id = str(document_id_raw).split("_", 1)[0].rstrip(".")
        roots = document_id_or_roots
    else:
        document_id = document_id_or_roots
        roots = volume_roots

    if "dbutils" not in globals():
        raise RuntimeError("Databricks dbutils is required for pipeline-backed volume discovery.")

    matches = _lookup_target_docs(
        volume_paths=roots,
        target_document_ids=[document_id_raw],
        existing_doc_ids=set(),
        dbutils_client=dbutils,
    )
    if not matches:
        return f"{roots[0].rstrip('/')}/{document_id}", roots[0], False

    match = matches[0]
    volume_path = match["volume_path"]
    selected_root = next(
        (root for root in roots if volume_path.startswith(f"{root.rstrip('/')}/")),
        roots[0],
    )
    return volume_path, selected_root, True

In [ ]:
import os as _os

_os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Using OUTPUT_DIR:", OUTPUT_DIR)

# Wire IBAT resolver for batch runs; falls back to None outside Databricks.
IBAT_TABLE = "vgpd.prm_std_views.ibat_equipment_mst"
_batch_ibat_resolver = None
try:
    from fsr_v2.metadata_processor_v2 import _make_ibat_resolver
    _batch_ibat_resolver = _make_ibat_resolver(spark, IBAT_TABLE)
    print("IBAT resolver enabled for batch run:", IBAT_TABLE)
except (NameError, ImportError) as _e:
    print("IBAT resolver disabled (no Spark / import unavailable):", _e)

batch_mode = True  # True -> batch run, False -> single-case flow with selected_case_index

# Choose source for batch IDs:
# - "custom_doc_ids": run the full regression list below
# - "test_cases": run cases configured in test_cases (optionally filtered by index)
batch_source = "custom_doc_ids"

# Used only when batch_source == "test_cases"
batch_case_indices = []  # empty means run all test_cases

# Full regression document list
custom_doc_ids = [
    "cc9fe3d7-87cc-4687-9fe3-d787cc3687b3",
    "0be4a3ad-7395-41ea-8ec2-e7669c9c15aa",
    "3f5a1ea8-8f35-4e97-9a1e-a88f358e9768",
    "bdd56c7a-ebe5-4bf7-904a-5bb63091ba20",
    "4597a853-5ffb-40bb-b0be-bd6307b0acdc",
    "27314604-ed52-402f-921f-34737a048841",
    "4a56cb21-fd6b-4300-ac1b-62cf53b868a0",
    "11338269-9d7c-4865-bfcf-1a4db9014acf",
    "af693a98-1e5c-499d-aa10-cccc54885c64",
    "b896cb9f-b70e-48c5-b9b1-477aa18bf03a",
    "32689520-afed-4dd4-a895-20afed7dd4d2",
    "b25c94da-d954-4283-9c94-dad954a28307",
    "b1cdbc80-364f-4240-8dbc-80364f1240fa",
    "b775cf29-8b42-4a83-af21-53075fef0802",
    "9e279e5a-a24e-4ebd-a79e-5aa24efebd04",
    "cdd0cca4-93ba-43b0-98e5-3d1ea2312c19",
    "69dfe261-34b0-4740-ab89-498e7d0072df",
    "d9b6c08b-d098-4939-a549-d113964e3150",
    "796f4d53-a8ad-42e1-af4d-53a8add2e1a4",
    "5b688732-39f2-48d2-a887-3239f258d28b",
    "fcb1511e-596a-4a56-b151-1e596afa569c",
    "b7b347fd-0b59-4c67-b609-9ad2c35105cc",
    "35803273-2440-4d36-87fe-e45f7f0e5467_605011422-40815-270t483-final_master_report",
    "806975e9-062b-4da3-9ae7-5881a5daffd1_204049598-39387-297652-final_master_report",
    "3e91b865-2aaa-4233-ae4a-3e15ba641f96_605007134-180498-270t483-final_master_report",
    "65f6535e-e26a-4e7a-bd41-6cdf4eaf1efa",
]

# Keep batch behavior aligned with single-case volume scanning.
batch_volume_roots = [
    "/Volumes/viud/ing_ud_fieldvision/fv_field_service_report",
    "/Volumes/vaid/ai_std_con_field_service_report/fsr_v2_test",
]

if batch_mode:
    if batch_source == "custom_doc_ids":
        case_records = [{"name": "custom-regression", "bug_id": "mixed", "document_id": d} for d in custom_doc_ids]
    else:
        if batch_case_indices:
            case_indices = batch_case_indices
        else:
            case_indices = list(range(len(test_cases)))
        case_records = [test_cases[i] for i in case_indices]

    batch_results = []

    for case in case_records:
        document_id_raw = case["document_id"]
        document_id = document_id_raw.split("_")[0].rstrip(".")
        volume_path, selected_root, found_in_any_root = _resolve_volume_path(document_id, batch_volume_roots)

        print("\n" + "=" * 90)
        print("Running case:", case.get("name", "custom-regression"))
        print("document_id_raw:", document_id_raw)
        print("document_id_normalized:", document_id)
        print("selected_volume_root:", selected_root)

        if not found_in_any_root:
            row = {
                "bug_id": case.get("bug_id"),
                "name": case.get("name"),
                "document_id_raw": document_id_raw,
                "document_id": document_id,
                "status": "skipped_missing_in_volume",
                "volume_path": volume_path,
                "volume_roots_scanned": batch_volume_roots,
            }
            batch_results.append(row)
            print("status: skipped_missing_in_volume |", volume_path)
            continue

        doc = {
            "document_id": document_id,
            "volume_path": volume_path,
        }

        try:
            parsed_local = parse_pymupdf(doc)
            ctx_local = SimpleNamespace(
                pages=parsed_local.pages,
                raw_pages=parsed_local.raw_pages,
                page_offsets=parsed_local.page_offsets,
                full_text=parsed_local.full_text,
                fields=fields,
                filename=parsed_local.filename,
            )
            result_local = preprocessor.preprocess(ctx_local, ibat_resolver=_batch_ibat_resolver)

            metadata_local = result_local.get("metadata", {}) or {}
            regions_local = result_local.get("regions", []) or []
            hints_local = result_local.get("hints", "") or ""

            artifact_local = {
                "bug_id": case.get("bug_id"),
                "name": case.get("name"),
                "document_id_raw": document_id_raw,
                "document_id": document_id,
                "volume_path": volume_path,
                "selected_volume_root": selected_root,
                "run_utc": datetime.now(timezone.utc).isoformat(),
                "page_count": len(parsed_local.pages),
                "char_count": len(parsed_local.full_text),
                "metadata": metadata_local,
                "hints": hints_local,
                "regions": regions_local,
            }

            out_path = f"{OUTPUT_DIR}/{document_id}.json"
            with open(out_path, "w") as f:
                f.write(json.dumps(artifact_local, indent=2))

            row = {
                "bug_id": case.get("bug_id"),
                "name": case.get("name"),
                "document_id_raw": document_id_raw,
                "document_id": document_id,
                "status": "ok",
                "volume_path": volume_path,
                "output_path": out_path,
                "primary_esn": metadata_local.get("primary_esn"),
                "primary_equip_type": metadata_local.get("primary_equip_type"),
                "region_count": len(regions_local),
            }
            batch_results.append(row)
            print("status: ok |", out_path)
            print("primary:", row["primary_equip_type"], row["primary_esn"], "| regions:", row["region_count"])

        except Exception as e:
            row = {
                "bug_id": case.get("bug_id"),
                "name": case.get("name"),
                "document_id_raw": document_id_raw,
                "document_id": document_id,
                "status": "error",
                "volume_path": volume_path,
                "error": str(e),
            }
            batch_results.append(row)
            print("status: error |", str(e))

    summary_path = f"{OUTPUT_DIR}/_batch_summary.json"
    with open(summary_path, "w") as f:
        f.write(json.dumps(batch_results, indent=2))

    print("\n" + "=" * 90)
    print("Batch summary path:", summary_path)
    print(json.dumps(batch_results, indent=2))

Using OUTPUT_DIR: /home/u560060992/dbx/fsr-preprocessor-v2-output

Running case: custom-regression
document_id_raw: cc9fe3d7-87cc-4687-9fe3-d787cc3687b3
document_id_normalized: cc9fe3d7-87cc-4687-9fe3-d787cc3687b3
selected_volume_root: /Volumes/viud/ing_ud_fieldvision/fv_field_service_report
status: skipped_missing_in_volume | /Volumes/viud/ing_ud_fieldvision/fv_field_service_report/cc9fe3d7-87cc-4687-9fe3-d787cc3687b3

Running case: custom-regression
document_id_raw: 0be4a3ad-7395-41ea-8ec2-e7669c9c15aa
document_id_normalized: 0be4a3ad-7395-41ea-8ec2-e7669c9c15aa
selected_volume_root: /Volumes/viud/ing_ud_fieldvision/fv_field_service_report
status: skipped_missing_in_volume | /Volumes/viud/ing_ud_fieldvision/fv_field_service_report/0be4a3ad-7395-41ea-8ec2-e7669c9c15aa

Running case: custom-regression
document_id_raw: bdd56c7a-ebe5-4bf7-904a-5bb63091ba20
document_id_normalized: bdd56c7a-ebe5-4bf7-904a-5bb63091ba20
selected_volume_root: /Volumes/viud/ing_ud_fieldvision/fv_field_service_

In [ ]:
# Ensure newly reported 8/12 docs are always included in batch regression runs.
new_0812_doc_ids = [
    "b896cb9f-b70e-48c5-b9b1-477aa18bf03a",
    "b1cdbc80-364f-4240-8dbc-80364f1240fa",
    "b25c94da-d954-4283-9c94-dad954a28307",
    "32689520-afed-4dd4-a895-20afed7dd4d2",
]

if "custom_doc_ids" not in globals():
    custom_doc_ids = []

custom_doc_ids = list(dict.fromkeys(list(custom_doc_ids) + new_0812_doc_ids))
print("8/12 docs merged into custom_doc_ids")
print("custom_doc_ids count:", len(custom_doc_ids))

In [ ]:
# De-duplicate custom doc IDs while preserving list order.
custom_doc_ids = list(dict.fromkeys(custom_doc_ids))
print(f"custom_doc_ids count after de-dup: {len(custom_doc_ids)}")

## Run Order

Run cells top-to-bottom:
1. Install dependency
2. (Optional) restart Python runtime
3. Import/setup repo path
4. Configure inputs and fields
5. Parse PDF
6. Run preprocessor
7. Region check for target ESN
8. Optional snippets
9. Save artifact JSON
10. (Optional) batch runner cell

Batch runner usage:
- Keep `batch_mode = False` for normal single-case runs using `selected_case_index`.
- Set `batch_mode = True` to run multiple cases in one shot.
- Use `batch_case_indices = [..]` for specific cases; keep it empty to run all configured cases.

If parsing fails on Databricks, confirm the `volume_path` is mounted and readable by the cluster.

## Validation Cells (Post-Run)

These cells validate preprocessor output quality for the selected case and optional batch runs.

They check:
- Region coverage/overlap continuity against parsed text
- Metadata contract presence and simple shape checks
- ESN consistency between document metadata and region metadata
- Batch summary health (if batch runner was used)

In [ ]:
# Validation 1: Region coverage and overlap checks

if 'parsed' not in globals() or 'regions' not in globals():
    print('Run parse + preprocessor cells first (parsed/regions missing).')
else:
    text_len = len(getattr(parsed, 'full_text', '') or '')
    normalized_regions = []
    for i, r in enumerate(regions or []):
        try:
            s = int(r.get('start', -1))
            e = int(r.get('end', -1))
        except Exception:
            s, e = -1, -1
        normalized_regions.append((i, s, e, r))

    invalid_bounds = [x for x in normalized_regions if x[1] < 0 or x[2] < x[1]]
    out_of_range = [x for x in normalized_regions if x[2] > text_len]

    sorted_regions = sorted(normalized_regions, key=lambda x: (x[1], x[2]))
    overlaps = []
    gaps = []
    for curr, nxt in zip(sorted_regions, sorted_regions[1:]):
        _, cs, ce, _ = curr
        _, ns, ne, _ = nxt
        if ns <= ce:
            overlaps.append((curr, nxt))
        elif ns > ce + 1:
            gaps.append((ce + 1, ns - 1))

    covered_chars = 0
    for _, s, e, _ in sorted_regions:
        covered_chars += max(0, e - s + 1)

    print('text_len:', text_len)
    print('region_count:', len(sorted_regions))
    print('invalid_bounds:', len(invalid_bounds))
    print('out_of_range:', len(out_of_range))
    print('overlap_pairs:', len(overlaps))
    print('gap_segments:', len(gaps))
    print('covered_chars (naive sum):', covered_chars)

    if invalid_bounds:
        print('\nSample invalid bounds (up to 3):')
        for x in invalid_bounds[:3]:
            print({'region_index': x[0], 'start': x[1], 'end': x[2]})

    if out_of_range:
        print('\nSample out-of-range regions (up to 3):')
        for x in out_of_range[:3]:
            print({'region_index': x[0], 'start': x[1], 'end': x[2], 'text_len': text_len})

    if overlaps:
        print('\nSample overlap pairs (up to 3):')
        for left, right in overlaps[:3]:
            print({
                'left': {'idx': left[0], 'start': left[1], 'end': left[2]},
                'right': {'idx': right[0], 'start': right[1], 'end': right[2]},
            })

    if gaps:
        print('\nSample gap segments (up to 3):', gaps[:3])

In [ ]:
# Validation 2: Metadata contract and ESN consistency checks

if 'metadata' not in globals() or 'regions' not in globals():
    print('Run preprocessor cell first (metadata/regions missing).')
else:
    required_keys = [
        'document_name',
        'primary_esn',
        'primary_equip_type',
        'gt_esn',
        'gen_esn',
        'st_esn',
        'all_esns',
        'inactive_esns',
        'outage_start_date',
        'outage_end_date',
        'report_issued_date',
    ]

    missing = [k for k in required_keys if k not in metadata]
    print('required_keys_missing:', missing)

    def _split_csv(v):
        return [x.strip() for x in (v or '').split(',') if x.strip()]

    all_esns = set(_split_csv(metadata.get('all_esns', '')))
    inactive_esns_set = set(_split_csv(metadata.get('inactive_esns', '')))

    region_esns = set()
    region_types = set()
    for r in regions or []:
        rm = (r or {}).get('metadata') or {}
        esn = (rm.get('primary_esn') or '').strip()
        et = (rm.get('primary_equip_type') or '').strip()
        if esn:
            region_esns.add(esn)
        if et:
            region_types.add(et)

    print('metadata.primary_esn:', metadata.get('primary_esn', ''))
    print('metadata.gt_esn:', metadata.get('gt_esn', ''))
    print('metadata.gen_esn:', metadata.get('gen_esn', ''))
    print('metadata.st_esn:', metadata.get('st_esn', ''))
    print('metadata.all_esns(list):', sorted(all_esns))
    print('metadata.inactive_esns(list):', sorted(inactive_esns_set))
    print('region_esns(list):', sorted(region_esns))
    print('region_types(list):', sorted(region_types))

    all_not_in_regions = sorted(all_esns - region_esns)
    regions_not_in_all = sorted(region_esns - all_esns)

    print('all_esns_not_in_regions:', all_not_in_regions)
    print('region_esns_not_in_all_esns:', regions_not_in_all)

    active_esns_from_all = sorted(all_esns - inactive_esns_set)
    print('active_esns_from_all_minus_inactive:', active_esns_from_all)

In [ ]:
# Validation 3: Case expectation check (when expected_esns_for_debug is configured)

if 'expected_esns_for_debug' not in globals() or 'metadata' not in globals():
    print('Run config + preprocessor cells first.')
else:
    expected = [x.strip() for x in (expected_esns_for_debug or []) if x and x.strip()]
    observed_all = [x.strip() for x in (metadata.get('all_esns', '') or '').split(',') if x.strip()]
    observed_set = set(observed_all)
    expected_set = set(expected)

    print('expected_esns_for_debug:', expected)
    print('observed_all_esns:', sorted(observed_set))

    if not expected:
        print('No expected ESNs configured for this case; skipping pass/fail check.')
    else:
        missing_expected = sorted(expected_set - observed_set)
        unexpected_extra = sorted(observed_set - expected_set)
        print('missing_expected_esns:', missing_expected)
        print('unexpected_extra_esns:', unexpected_extra)
        print('expectation_passed:', len(missing_expected) == 0)

In [ ]:
# Validation 4 (optional): Batch summary health check

import json as _json
import os as _os

if 'OUTPUT_DIR' not in globals():
    print('OUTPUT_DIR not found. Run save/batch cells first if needed.')
else:
    summary_path = f"{OUTPUT_DIR}/_batch_summary.json"
    if not _os.path.exists(summary_path):
        print('No batch summary found at:', summary_path)
    else:
        with open(summary_path, 'r') as f:
            batch_rows = _json.load(f)

        total = len(batch_rows)
        by_status = {}
        by_bug = {}
        for row in batch_rows:
            st = row.get('status', 'unknown')
            by_status[st] = by_status.get(st, 0) + 1
            bug = row.get('bug_id', 'unknown')
            by_bug[bug] = by_bug.get(bug, 0) + 1

        print('batch_summary_path:', summary_path)
        print('total_cases:', total)
        print('status_counts:', by_status)
        print('bug_id_counts:', by_bug)

        failures = [r for r in batch_rows if r.get('status') not in ('ok',)]
        if failures:
            print('\nNon-ok rows (up to 10):')
            for r in failures[:10]:
                print({
                    'case_index': r.get('case_index'),
                    'bug_id': r.get('bug_id'),
                    'name': r.get('name'),
                    'document_id': r.get('document_id'),
                    'status': r.get('status'),
                    'error': r.get('error', ''),
                })
        else:
            print('All batch rows are ok.')

In [ ]:
# Validation 5: Final PASS/FAIL summary

summary = {
    "ready": True,
    "checks": {},
    "notes": [],
}

required_globals = ["parsed", "regions", "metadata"]
missing_globals = [g for g in required_globals if g not in globals()]
if missing_globals:
    summary["ready"] = False
    summary["notes"].append(f"Missing required variables: {missing_globals}. Run parse + preprocessor + validation cells first.")

if summary["ready"]:
    text_len = len(getattr(parsed, "full_text", "") or "")

    normalized = []
    for r in regions or []:
        try:
            s = int((r or {}).get("start", -1))
            e = int((r or {}).get("end", -1))
        except Exception:
            s, e = -1, -1
        normalized.append((s, e, r))

    normalized.sort(key=lambda x: (x[0], x[1]))

    invalid_bounds = any(s < 0 or e < s for s, e, _ in normalized)
    out_of_range = any(e > text_len for s, e, _ in normalized)

    overlap_count = 0
    gap_count = 0
    for (cs, ce, _), (ns, ne, _) in zip(normalized, normalized[1:]):
        if ns <= ce:
            overlap_count += 1
        elif ns > ce + 1:
            gap_count += 1

    required_keys = {
        "document_name",
        "primary_esn",
        "primary_equip_type",
        "gt_esn",
        "gen_esn",
        "st_esn",
        "all_esns",
        "inactive_esns",
        "outage_start_date",
        "outage_end_date",
        "report_issued_date",
    }
    missing_keys = sorted([k for k in required_keys if k not in metadata])

    def _split_csv(v):
        return [x.strip() for x in (v or "").split(",") if x.strip()]

    all_esns = set(_split_csv(metadata.get("all_esns", "")))
    region_esns = set()
    for _, _, r in normalized:
        rm = (r or {}).get("metadata") or {}
        esn = (rm.get("primary_esn") or "").strip()
        if esn:
            region_esns.add(esn)

    expected = [x.strip() for x in (globals().get("expected_esns_for_debug", []) or []) if x and x.strip()]
    expected_set = set(expected)
    expected_missing = sorted(expected_set - all_esns) if expected else []

    has_nonempty_text = text_len > 0
    summary["checks"]["has_nonempty_text"] = has_nonempty_text
    summary["checks"]["has_regions"] = len(normalized) > 0
    summary["checks"]["region_bounds_valid"] = not invalid_bounds
    summary["checks"]["region_bounds_in_text_range"] = not out_of_range
    summary["checks"]["no_region_overlaps"] = overlap_count == 0
    summary["checks"]["no_region_gaps"] = gap_count == 0
    summary["checks"]["metadata_required_keys_present"] = len(missing_keys) == 0
    summary["checks"]["all_esns_contains_region_esns"] = region_esns.issubset(all_esns)
    summary["checks"]["expected_esns_present"] = (len(expected_missing) == 0) if expected else True

    summary["notes"].append(f"region_count={len(normalized)} text_len={text_len} overlaps={overlap_count} gaps={gap_count}")
    if not has_nonempty_text:
        summary["notes"].append("parsed.full_text is empty. This usually means parse was skipped or source file is missing in volume path.")
    if missing_keys:
        summary["notes"].append(f"missing_metadata_keys={missing_keys}")
    if expected and expected_missing:
        summary["notes"].append(f"expected_esns_missing={expected_missing}")

all_pass = summary["ready"] and all(summary["checks"].values())
summary["result"] = "PASS" if all_pass else "FAIL"

print("\n=== PREPROCESSOR VALIDATION SUMMARY ===")
print("result:", summary["result"] )
print("ready:", summary["ready"] )
print("\nchecks:")
for k, v in summary["checks"].items():
    print(f"  - {k}: {v}")
print("\nnotes:")
for n in summary["notes"]:
    print("  -", n)

summary

In [ ]:
# Validation 6: Region metadata schema validation (new schema)

schema_summary = {
    "ready": True,
    "errors": [],
    "warnings": [],
    "counts": {
        "regions": 0,
        "missing_fields": 0,
        "invalid_enum": 0,
        "invalid_shape": 0,
    },
}

if "regions" not in globals():
    schema_summary["ready"] = False
    schema_summary["errors"].append("regions variable is missing. Run preprocessor cell first.")
else:
    allowed_esn_confidence = {"highest", "high", "low", "fallback", "none"}
    allowed_esn_source = {
        "local_header",
        "parent_inherit",
        "single_type",
        "ibat_train",
        "neighbor_gap",
        "doc_primary",
        "none",
    }
    allowed_region_source = {
        "section_span",
        "front_matter",
        "gap_fallback",
        "trailing",
        "synthetic",
    }
    allowed_equip_type_source = {
        "heading",
        "parent_inherit",
        "neighbor_consensus",
        "doc_primary",
        "none",
        "neighbor_single",
    }

    required_meta_fields = [
        "esn_confidence",
        "esn_source",
        "equip_type_source",
        "region_source",
        "fallback_chain",
        "section_path",
    ]

    def _norm_enum(v):
        s = str(v or "").strip()
        if "." in s:
            s = s.rsplit(".", 1)[-1]
        return s.lower()

    schema_summary["counts"]["regions"] = len(regions or [])
    parsed_text_len = len(getattr(globals().get("parsed", None), "full_text", "") or "")

    # Empty parsed text is an environment/data-availability case; mark as SKIP.
    skip_due_to_empty = parsed_text_len == 0
    if skip_due_to_empty:
        schema_summary["warnings"].append(
            "Parsed text is empty. Parse likely skipped or source file missing in volume path. Marking schema validation as SKIP."
        )

    if not skip_due_to_empty:
        # Zero-region output should not pass silently.
        if schema_summary["counts"]["regions"] == 0 and parsed_text_len > 0:
            schema_summary["errors"].append(
                f"No regions emitted for non-empty parsed text (chars={parsed_text_len})."
            )

        for idx, region in enumerate(regions or []):
            meta = (region or {}).get("metadata") or {}

            # Required field presence
            missing = [k for k in required_meta_fields if k not in meta]
            if missing:
                schema_summary["counts"]["missing_fields"] += 1
                schema_summary["errors"].append({
                    "region_index": idx,
                    "type": "missing_fields",
                    "fields": missing,
                })
                continue

            # Enum validations (accept raw enum-like strings such as EsnConfidence.NONE)
            ec_raw = str(meta.get("esn_confidence", "")).strip()
            es_raw = str(meta.get("esn_source", "")).strip()
            rs_raw = str(meta.get("region_source", "")).strip()
            ets_raw = str(meta.get("equip_type_source", "")).strip()

            ec = _norm_enum(ec_raw)
            es = _norm_enum(es_raw)
            rs = _norm_enum(rs_raw)
            ets = _norm_enum(ets_raw)

            invalid_enum_fields = []
            if ec not in allowed_esn_confidence:
                invalid_enum_fields.append({"field": "esn_confidence", "value": ec_raw, "normalized": ec})
            if es not in allowed_esn_source:
                invalid_enum_fields.append({"field": "esn_source", "value": es_raw, "normalized": es})
            if rs not in allowed_region_source:
                invalid_enum_fields.append({"field": "region_source", "value": rs_raw, "normalized": rs})
            if ets not in allowed_equip_type_source:
                invalid_enum_fields.append({"field": "equip_type_source", "value": ets_raw, "normalized": ets})

            if invalid_enum_fields:
                schema_summary["counts"]["invalid_enum"] += 1
                schema_summary["errors"].append({
                    "region_index": idx,
                    "type": "invalid_enum",
                    "details": invalid_enum_fields,
                })

            # Shape/type checks
            shape_issues = []
            fallback_chain = meta.get("fallback_chain")
            section_path = meta.get("section_path")

            if not isinstance(fallback_chain, list):
                shape_issues.append("fallback_chain_not_list")
            else:
                if not all(isinstance(x, str) for x in fallback_chain):
                    shape_issues.append("fallback_chain_non_string_item")

            if not isinstance(section_path, list):
                shape_issues.append("section_path_not_list")
            else:
                if len(section_path) > 5:
                    shape_issues.append("section_path_length_gt_5")
                if not all(isinstance(x, str) for x in section_path):
                    shape_issues.append("section_path_non_string_item")

            # Contract consistency checks
            primary_esn = (meta.get("primary_esn") or "").strip()
            if ec == "none" and primary_esn:
                schema_summary["warnings"].append({
                    "region_index": idx,
                    "type": "esn_confidence_none_but_primary_esn_present",
                    "primary_esn": primary_esn,
                })

            if ec != "none" and not primary_esn:
                schema_summary["warnings"].append({
                    "region_index": idx,
                    "type": "esn_confidence_non_none_but_primary_esn_missing",
                    "esn_confidence": ec_raw,
                })

            if shape_issues:
                schema_summary["counts"]["invalid_shape"] += 1
                schema_summary["errors"].append({
                    "region_index": idx,
                    "type": "invalid_shape",
                    "details": shape_issues,
                })

schema_ok = schema_summary["ready"] and len(schema_summary["errors"]) == 0
if schema_ok and "regions" in globals() and len(getattr(globals().get("parsed", None), "full_text", "") or "") == 0:
    schema_summary["result"] = "SKIP"
else:
    schema_summary["result"] = "PASS" if schema_ok else "FAIL"

print("\n=== REGION METADATA SCHEMA VALIDATION ===")
print("result:", schema_summary["result"] )
print("ready:", schema_summary["ready"] )
print("counts:", schema_summary["counts"] )
print("warning_count:", len(schema_summary["warnings"]))

if schema_summary["errors"]:
    print("\nErrors (up to 10):")
    for e in schema_summary["errors"][:10]:
        print(" -", e)

if schema_summary["warnings"]:
    print("\nWarnings (up to 10):")
    for w in schema_summary["warnings"][:10]:
        print(" -", w)

schema_summary

## Bug 2 Verification: Section Header Detection Across Page Join

This check validates whether headers like `2 Turbine` are detected when there is no newline before the header token (simulating pypdf2 page-join edge cases).

Run the next cell and check:
- `with_newline` should detect `2 Turbine`
- `without_newline` should also detect `2 Turbine` for bug 2 to be considered fixed

In [7]:
import re

# This check validates the actual candidate-collection path, not regex-only matching.
from common.fsr_v2.preprocessor_v2 import FSRV2Preprocessor, SECTION_HDR

pre = FSRV2Preprocessor()

# Control: newline before header (classic regex path should work)
page1_a = "end prev page\n"
page2_a = "2 Turbine\n2.1 Scope\n"
full_text_a = page1_a + page2_a
raw_pages_a = [page1_a, page2_a]
page_offsets_a = [0, len(page1_a)]

# Edge: no newline at page join (reported bug condition)
page1_b = "end prev page"
page2_b = "2 Turbine\n2.1 Scope\n"
full_text_b = page1_b + page2_b
raw_pages_b = [page1_b, page2_b]
page_offsets_b = [0, len(page1_b)]

# Show raw regex behavior for transparency
regex_hits_a = [m.group(0).strip() for m in re.finditer(SECTION_HDR, full_text_a)]
regex_hits_b = [m.group(0).strip() for m in re.finditer(SECTION_HDR, full_text_b)]

# Run candidate collection (page-aware) used by preprocess
cands_a, _ = pre._collect_heading_candidates(
    full_text_a,
    raw_pages_a,
    pages=raw_pages_a,
    page_offsets=page_offsets_a,
)
cands_b, _ = pre._collect_heading_candidates(
    full_text_b,
    raw_pages_b,
    pages=raw_pages_b,
    page_offsets=page_offsets_b,
)

section_hits_a = [c.heading_text for c in cands_a if c.heading_type.value == "SECTION_HDR" and "2 Turbine" in c.heading_text]
section_hits_b = [c.heading_text for c in cands_b if c.heading_type.value == "SECTION_HDR" and "2 Turbine" in c.heading_text]

print("Bug 2 synthetic check results:")
print(f"- regex with_newline: {regex_hits_a}")
print(f"- regex without_newline: {regex_hits_b}")
print(f"- collector with_newline: {section_hits_a}")
print(f"- collector without_newline: {section_hits_b}")

with_newline_ok = len(section_hits_a) > 0
without_newline_ok = len(section_hits_b) > 0

print("\nVerdict:")
print(f"- with_newline detected: {with_newline_ok}")
print(f"- without_newline detected: {without_newline_ok}")

if with_newline_ok and without_newline_ok:
    print("PASS: Bug 2 synthetic check passed.")
else:
    print("FAIL: Bug 2 synthetic check did not pass fully.")

Bug 2 synthetic check results:
- regex with_newline: ['2 Turbine']
- regex without_newline: []
- collector with_newline: ['2 Turbine']
- collector without_newline: ['2 Turbine']

Verdict:
- with_newline detected: True
- without_newline detected: True
PASS: Bug 2 synthetic check passed.


## Validation 7: 8/12 Doc Targeted Sanity Checks

Runs targeted checks for newly reported docs:
- b896cb9f-b70e-48c5-b9b1-477aa18bf03a
- b1cdbc80-364f-4240-8dbc-80364f1240fa
- b25c94da-d954-4283-9c94-dad954a28307
- 32689520-afed-4dd4-a895-20afed7dd4d2

These checks read saved JSON artifacts from OUTPUT_DIR and flag likely mismatch patterns for quick review.

In [ ]:
import json
import os

TARGET_DOCS_0812 = [
    "b896cb9f-b70e-48c5-b9b1-477aa18bf03a",
    "b1cdbc80-364f-4240-8dbc-80364f1240fa",
    "b25c94da-d954-4283-9c94-dad954a28307",
    "32689520-afed-4dd4-a895-20afed7dd4d2",
]

summary_0812 = {
    "output_dir": OUTPUT_DIR,
    "docs": {},
    "missing_artifacts": [],
}

def _load_artifact(doc_id: str):
    path = os.path.join(OUTPUT_DIR, f"{doc_id}.json")
    if not os.path.exists(path):
        return None, path
    with open(path, "r") as artifact_file:
        return json.load(artifact_file), path

def _has_orphan_level_minus_one(meta: dict) -> bool:
    section_path = [str(value).strip() for value in (meta.get("section_path") or [])]
    if not section_path:
        return False
    top = section_path[0].upper()
    is_equipment_root_only = len(section_path) == 1 and top in {"GAS TURBINE", "GENERATOR", "STEAM TURBINE"}
    no_esn = not str(meta.get("primary_esn") or "").strip()
    esn_source = str(meta.get("esn_source") or "").strip().lower()
    return is_equipment_root_only and no_esn and esn_source in {"none", "", "doc_primary"}

for doc_id in TARGET_DOCS_0812:
    artifact, artifact_path = _load_artifact(doc_id)
    if artifact is None:
        summary_0812["missing_artifacts"].append({"doc_id": doc_id, "path": artifact_path})
        continue

    regions = artifact.get("regions") or []
    flagged = {
        "region_count": len(regions),
        "possible_inheritance_mismatch": 0,
        "possible_orphan_level_minus_one": 0,
    }

    for region in regions:
        meta = (region or {}).get("metadata") or {}
        section_path = [str(value) for value in (meta.get("section_path") or [])]
        equip_type = str(meta.get("primary_equip_type") or "")

        if doc_id == "b896cb9f-b70e-48c5-b9b1-477aa18bf03a":
            under_steam = any("STEAM TURBINE" in item.upper() for item in section_path)
            turbine_leaf = any("TURBINE" in item.upper() for item in section_path[-1:]) if section_path else False
            if under_steam and turbine_leaf and equip_type.strip().lower() == "gas turbine":
                flagged["possible_inheritance_mismatch"] += 1

        if doc_id in {
            "b25c94da-d954-4283-9c94-dad954a28307",
            "32689520-afed-4dd4-a895-20afed7dd4d2",
        } and _has_orphan_level_minus_one(meta):
            flagged["possible_orphan_level_minus_one"] += 1

    summary_0812["docs"][doc_id] = {
        "artifact_path": artifact_path,
        "checks": flagged,
    }

print("=== 8/12 TARGET DOC SANITY CHECK ===")
print("output_dir:", summary_0812["output_dir"])
print("missing_artifacts:", len(summary_0812["missing_artifacts"]))
for missing in summary_0812["missing_artifacts"]:
    print(" -", missing["doc_id"], "->", missing["path"])

for doc_id, row in summary_0812["docs"].items():
    checks = row["checks"]
    print(f"\n{doc_id}")
    print(" - region_count:", checks["region_count"])
    print(" - possible_inheritance_mismatch:", checks["possible_inheritance_mismatch"])
    print(" - possible_orphan_level_minus_one:", checks["possible_orphan_level_minus_one"])

summary_0812

## Validation 8: Generator Keyword Expansion Synthetic Check

Synthetic check for Generator keyword coverage candidates:
- Electrical
- Electrification
- DC leakage

Use this after updating heading-detection keywords to confirm these headings are recognized in candidate collection.

In [8]:
from common.fsr_v2.preprocessor_v2 import FSRV2Preprocessor

pre_kw = FSRV2Preprocessor()

# Synthetic text with Generator-related headings.
synthetic_text = (
    "1 Turbine\n"
    "1.1 Scope\n"
    "2 Generator\n"
    "2.1 Electrical Inspection\n"
    "2.2 Electrification Check\n"
    "2.3 DC leakage trend\n"
)

cands_kw, _ = pre_kw._collect_heading_candidates(
    synthetic_text,
    raw_pages=[synthetic_text],
    pages=[synthetic_text],
    page_offsets=[0],
)

# Surface heading candidates for inspection.
candidate_rows = []
for c in cands_kw:
    candidate_rows.append({
        "heading_text": c.heading_text,
        "heading_type": c.heading_type.value,
        "equipment_type": c.equipment_type,
        "level": c.level,
    })

hits = [
    row for row in candidate_rows
    if any(k in row["heading_text"].lower() for k in ["electrical", "electrification", "dc leakage"])
]

print("=== GENERATOR KEYWORD SYNTHETIC CHECK ===")
print("total_candidates:", len(candidate_rows))
print("keyword_hits:", len(hits))
for h in hits:
    print(" -", h)

if len(hits) == 0:
    print("\nResult: No keyword-based headings detected yet. Expected to change after keyword expansion is implemented.")
else:
    print("\nResult: Keyword-based headings detected.")

{
    "total_candidates": len(candidate_rows),
    "keyword_hits": hits,
}

=== GENERATOR KEYWORD SYNTHETIC CHECK ===
total_candidates: 5
keyword_hits: 3
 - {'heading_text': '2.1 Electrical', 'heading_type': 'SUBSEC', 'equipment_type': 'Generator', 'level': 1}
 - {'heading_text': '2.2 Electrification', 'heading_type': 'SUBSEC', 'equipment_type': 'Generator', 'level': 1}
 - {'heading_text': '2.3 DC leakage', 'heading_type': 'SUBSEC', 'equipment_type': 'Generator', 'level': 1}

Result: Keyword-based headings detected.


{'total_candidates': 5,
 'keyword_hits': [{'heading_text': '2.1 Electrical',
   'heading_type': 'SUBSEC',
   'equipment_type': 'Generator',
   'level': 1},
  {'heading_text': '2.2 Electrification',
   'heading_type': 'SUBSEC',
   'equipment_type': 'Generator',
   'level': 1},
  {'heading_text': '2.3 DC leakage',
   'heading_type': 'SUBSEC',
   'equipment_type': 'Generator',
   'level': 1}]}

## Validation 9: 8-19 TOC Cross-check Targeted Sanity Checks (commit `7ec453a`)

Reads saved JSON artifacts from `OUTPUT_DIR` to verify the two targeted regressions
addressed by the 8-19 TOC cross-check fix:

- `4597a853-5ffb-40bb-b0be-bd6307b0acdc` — the whole doc was previously collapsing
  into a single synthetic `shared` region because `3 Generator` at page 11 landed
  inside the old `toc_cutoff` (first 5% of doc). After the fix, `3 Generator`
  must survive via TOC cross-check and the doc-level primary must resolve to
  `Generator / 338X447`.
- `b775cf29-8b42-4a83-af21-53075fef0802` — OCR artifact `3.8\nPosition` was leaking
  into `section_path`. After the fix, no region should carry a `Position` element
  in its `section_path` (TOC has no matching `3.8 Position` entry).

Run the batch cell first so artifacts exist in `OUTPUT_DIR`.

In [ ]:
import json
import os

TARGET_DOCS_0819 = {
    "4597a853-5ffb-40bb-b0be-bd6307b0acdc": {
        "check_name": "toc_crosscheck_restores_early_section_hdr",
        "expected_primary_equip_type": "Generator",
        "expected_primary_esn": "338X447",
        "expected_section_path_leaf": "3 Generator",
        "min_expected_region_count": 5,
    },
    "b775cf29-8b42-4a83-af21-53075fef0802": {
        "check_name": "subsec_generic_ocr_artifact_filtered",
        "forbidden_section_path_substr": "position",
        "expected_primary_equip_type": "Gas Turbine",
        "expected_primary_esn": "297651",
    },
}

summary_0819 = {
    "output_dir": OUTPUT_DIR,
    "results": {},
    "missing_artifacts": [],
}

def _load_artifact_0819(doc_id: str):
    path = os.path.join(OUTPUT_DIR, f"{doc_id}.json")
    if not os.path.exists(path):
        return None, path
    with open(path, "r") as artifact_file:
        return json.load(artifact_file), path

for doc_id, expectations in TARGET_DOCS_0819.items():
    artifact, artifact_path = _load_artifact_0819(doc_id)
    if artifact is None:
        summary_0819["missing_artifacts"].append({"doc_id": doc_id, "path": artifact_path})
        continue

    meta = artifact.get("metadata") or {}
    regions = artifact.get("regions") or []

    doc_report = {
        "artifact_path": artifact_path,
        "check_name": expectations["check_name"],
        "region_count": len(regions),
        "primary_equip_type": meta.get("primary_equip_type"),
        "primary_esn": meta.get("primary_esn"),
        "failures": [],
    }

    if "expected_primary_equip_type" in expectations:
        want = expectations["expected_primary_equip_type"]
        got = str(meta.get("primary_equip_type") or "")
        if got.strip().lower() != want.strip().lower():
            doc_report["failures"].append({
                "type": "primary_equip_type_mismatch",
                "want": want,
                "got": got,
            })

    if "expected_primary_esn" in expectations:
        want = str(expectations["expected_primary_esn"]).strip()
        got = str(meta.get("primary_esn") or "").strip()
        if got != want:
            doc_report["failures"].append({
                "type": "primary_esn_mismatch",
                "want": want,
                "got": got,
            })

    if "min_expected_region_count" in expectations:
        want = int(expectations["min_expected_region_count"])
        if len(regions) < want:
            doc_report["failures"].append({
                "type": "region_count_too_low",
                "want_at_least": want,
                "got": len(regions),
            })

    if "expected_section_path_leaf" in expectations:
        want = expectations["expected_section_path_leaf"]
        found = False
        for region in regions:
            rm = (region or {}).get("metadata") or {}
            sp = [str(x) for x in (rm.get("section_path") or [])]
            if sp and sp[-1] == want:
                found = True
                break
        if not found:
            doc_report["failures"].append({
                "type": "section_path_leaf_missing",
                "want": want,
            })

    if "forbidden_section_path_substr" in expectations:
        needle = expectations["forbidden_section_path_substr"].lower()
        offending = []
        for idx, region in enumerate(regions):
            rm = (region or {}).get("metadata") or {}
            sp = [str(x) for x in (rm.get("section_path") or [])]
            for elem in sp:
                if needle in elem.lower():
                    offending.append({"region_index": idx, "section_path": sp})
                    break
        if offending:
            doc_report["failures"].append({
                "type": "forbidden_section_path_substr_present",
                "needle": needle,
                "count": len(offending),
                "sample": offending[:3],
            })

    doc_report["result"] = "PASS" if not doc_report["failures"] else "FAIL"
    summary_0819["results"][doc_id] = doc_report

print("=== 8-19 TOC CROSS-CHECK TARGET DOC SANITY (commit 7ec453a) ===")
print("output_dir:", summary_0819["output_dir"])
print("missing_artifacts:", len(summary_0819["missing_artifacts"]))
for missing in summary_0819["missing_artifacts"]:
    print(" -", missing["doc_id"], "->", missing["path"])

any_fail = False
for doc_id, row in summary_0819["results"].items():
    print(f"\n{doc_id} [{row['check_name']}] -> {row['result']}")
    print(" - region_count:", row["region_count"])
    print(" - primary_equip_type:", row["primary_equip_type"])
    print(" - primary_esn:", row["primary_esn"])
    if row["failures"]:
        any_fail = True
        for failure in row["failures"]:
            print(" - FAIL:", failure)

summary_0819["overall_result"] = "PASS" if (
    not any_fail and not summary_0819["missing_artifacts"]
) else "FAIL"
print("\noverall_result:", summary_0819["overall_result"])
summary_0819

## Validation 10: 8-19 TOC Cross-check Synthetic Check (commit `7ec453a`)

Two in-notebook synthetic checks that exercise the TOC cross-check helpers
directly (no need for saved artifacts):

- SECTION_HDR path: a bare `3 Generator` heading that lands inside the previous
  5% cutoff window survives when TOC has a matching `Generator` anchor.
- SUBSEC_GENERIC path: an OCR-artifact `3.8\nPosition` heading is dropped when
  the TOC's `3.8` entry has an unrelated title.

Also verifies the safety fallback: when the TOC parser returns fewer than
`_TOC_CROSSCHECK_MIN_ENTRIES` entries, both helpers accept the candidate
so we do not regress docs where TOC parsing itself is weak.

In [ ]:
from common.fsr_v2.preprocessor_v2 import (
    FSRV2Preprocessor,
    HeadingType,
    _TOC_CROSSCHECK_MIN_ENTRIES,
    _section_hdr_matches_toc,
    _subsec_generic_matches_toc,
)

pre_toc = FSRV2Preprocessor()

toc_rich = [
    ("1 Executive Summary", 3),
    ("2 Overview", 5),
    ("3 Generator", 11),
    ("3.1 Field Winding Test", 15),
    ("3.2 Rotor Inspection", 20),
    ("4 Support", 100),
]
toc_no_position = [
    ("1 Executive Summary", 3),
    ("2 Overview", 5),
    ("3 Turbine", 11),
    ("3.1 Compressor", 15),
    ("3.8 Turbine Section", 20),
    ("4 Generator", 100),
]
toc_with_position = [
    ("1 Executive Summary", 3),
    ("3 Turbine", 11),
    ("3.8 Position Alignment", 20),
    ("4 Generator", 100),
    ("5 More", 200),
]
weak_toc = [("Just one entry", 1)]

toc_anchors_rich = pre_toc._toc_equipment_anchors(toc_rich)
toc_anchors_no_position = pre_toc._toc_equipment_anchors(toc_no_position)

section_hdr_generator_kept = _section_hdr_matches_toc(
    "GENERATOR", toc_rich, toc_anchors_rich,
)
section_hdr_generator_dropped_no_gen_anchor = _section_hdr_matches_toc(
    "GENERATOR", toc_no_position, toc_anchors_no_position,
)
section_hdr_weak_toc_kept = _section_hdr_matches_toc(
    "GENERATOR", weak_toc, set(),
)

subsec_position_dropped = _subsec_generic_matches_toc(
    "3.8", "Position", toc_no_position,
)
subsec_position_kept = _subsec_generic_matches_toc(
    "3.8", "Position", toc_with_position,
)
subsec_weak_toc_kept = _subsec_generic_matches_toc(
    "3.8", "Position", weak_toc,
)

# End-to-end SECTION_HDR check: bare `3 Generator` early in a long doc.
toc_page = "\n".join([
    "Table of Contents",
    "1 Executive Summary ....... 3",
    "2 Overview .............. 5",
    "3 Generator ............. 11",
    "3.1 Field Winding Test .. 15",
    "3.2 Rotor Inspection .... 20",
    "3.3 Stator Test ......... 25",
    "4 Support .............. 100",
    "5 Attachments .......... 200",
])
title_page = "\n".join([
    "Field Service Report",
    "Generator ESN: 338X447",
    "Gas Turbine ESN: 298250",
    "Job Start Date: 12 Dec",
])
body = "\n".join(["3 Generator", "This section covers Generator inspection."])
padding = "\n".join(["irrelevant filler content"] * 3000)
long_text = "\n".join([title_page, toc_page, body, padding])

old_cutoff = len(long_text) // 20
toc_marker = long_text.find("Table of Contents")
generator_body_offset = long_text.find("3 Generator", toc_marker + 20)

candidates_long, _ = pre_toc._collect_heading_candidates(
    long_text,
    raw_pages=[long_text],
    pages=[long_text],
    page_offsets=[0],
    active_by_type={"Gas Turbine": ["298250"], "Generator": ["338X447"], "Steam Turbine": []},
)
section_hdrs_long = [
    c for c in candidates_long
    if c.heading_type == HeadingType.SECTION_HDR and "3 Generator" in c.heading_text
]
early_section_hdr_survives = len(section_hdrs_long) > 0

# End-to-end SUBSEC_GENERIC check: OCR artifact `3.8\nPosition` dropped.
b775_style = "\n".join([
    "Table of Contents",
    "1 Executive Summary ....... 3",
    "2 Overview .............. 5",
    "3 Turbine ............... 11",
    "3.1 Compressor .......... 15",
    "3.8 Turbine Section ..... 20",
    "4 Generator ............ 100",
    "5 Attachments .......... 200",
    "",
    "GAS TURBINE (298250 | SY0000001)",
    "3 Turbine",
    "Turbine content",
    "### 3.1 Compressor",
    "Compressor content",
    "3.8",
    "Position",
    "Table label OCR artifact content",
])

candidates_b775, _ = pre_toc._collect_heading_candidates(
    b775_style,
    raw_pages=[b775_style],
    pages=[b775_style],
    page_offsets=[0],
)
position_subsec_present = any(
    "position" in c.heading_text.lower()
    for c in candidates_b775
    if c.heading_type == HeadingType.SUBSEC
)
position_subsec_dropped = not position_subsec_present

print("=== 8-19 TOC CROSS-CHECK SYNTHETIC CHECK ===")
print(f"_TOC_CROSSCHECK_MIN_ENTRIES = {_TOC_CROSSCHECK_MIN_ENTRIES}")

print("\n[helper] _section_hdr_matches_toc:")
print(f"  - keeps GENERATOR when TOC has Generator anchor: {section_hdr_generator_kept}")
print(f"  - drops GENERATOR when TOC has no Generator anchor: {not section_hdr_generator_dropped_no_gen_anchor}")
print(f"  - safety fallback (weak TOC) keeps candidate: {section_hdr_weak_toc_kept}")

print("\n[helper] _subsec_generic_matches_toc:")
print(f"  - drops `3.8 Position` when TOC has unrelated 3.8 entry: {not subsec_position_dropped}")
print(f"  - keeps `3.8 Position` when TOC has matching 3.8 Position: {subsec_position_kept}")
print(f"  - safety fallback (weak TOC) keeps candidate: {subsec_weak_toc_kept}")

print("\n[end-to-end] SECTION_HDR early-body survival:")
print(f"  - total_chars={len(long_text)}  old_toc_cutoff(5%)={old_cutoff}")
print(f"  - `3 Generator` body offset={generator_body_offset}  inside_old_cutoff={generator_body_offset <= old_cutoff}")
print(f"  - `3 Generator` SECTION_HDR emitted after TOC cross-check: {early_section_hdr_survives}")

print("\n[end-to-end] SUBSEC_GENERIC OCR artifact filter:")
print(f"  - `3.8\\nPosition` SUBSEC dropped by TOC cross-check: {position_subsec_dropped}")

checks_passed = all([
    section_hdr_generator_kept,
    (not section_hdr_generator_dropped_no_gen_anchor),
    section_hdr_weak_toc_kept,
    (not subsec_position_dropped),
    subsec_position_kept,
    subsec_weak_toc_kept,
    early_section_hdr_survives,
    position_subsec_dropped,
])

print("\nVerdict:")
if checks_passed:
    print("PASS: 8-19 TOC cross-check synthetic check passed.")
else:
    print("FAIL: 8-19 TOC cross-check synthetic check did not pass fully.")

{
    "helper_section_hdr_keeps_when_anchor_present": section_hdr_generator_kept,
    "helper_section_hdr_drops_when_anchor_missing": not section_hdr_generator_dropped_no_gen_anchor,
    "helper_section_hdr_safety_fallback": section_hdr_weak_toc_kept,
    "helper_subsec_generic_drops_ocr_artifact": not subsec_position_dropped,
    "helper_subsec_generic_keeps_valid_match": subsec_position_kept,
    "helper_subsec_generic_safety_fallback": subsec_weak_toc_kept,
    "e2e_section_hdr_early_body_survives": early_section_hdr_survives,
    "e2e_subsec_generic_ocr_artifact_dropped": position_subsec_dropped,
    "all_passed": checks_passed,
}

## Validation 11: 8-19 Options A + B — multi-line drop and small front-matter guard

Two follow-on filters layered on top of the TOC cross-check to compensate for the
weak TOC extractor on real FSR docs:

- **Option A**: SUBSEC_GENERIC drops any match whose text contains `\n` (real
  headings never span lines; `3.8\nPosition` is a page-join OCR artifact).
- **Option B**: A small `toc_cutoff = min(len // 50, 5000)` base guard on both
  SECTION_HDR loops and SUBSEC_GENERIC. Keeps TOC/index lines in the first ~2%
  of the doc from being pulled as body headers, without trapping legitimate
  body sections in long docs.

These two together address the residual regressions observed on `4597a853`
(spurious `3 Generator` at char 1227) and `b775cf29` (`3.8\nPosition` slipping
through when the TOC extractor returns few entries).

Run the batch cell first so artifacts exist in `OUTPUT_DIR`.

In [ ]:
import json
import os

TARGET_DOCS_0819_AB = {
    "4597a853-5ffb-40bb-b0be-bd6307b0acdc": {
        "check_name": "front_matter_guard_drops_stub_section_hdr",
        "expected_primary_equip_type": "Generator",
        "expected_primary_esn": "338X447",
        "min_expected_region_count": 5,
        # Body `3 Generator` in the good baseline started at char 12324; the
        # 8-19 pre-A+B run pulled it at char 1227 (front-matter TOC line).
        # After A+B, no `3 Generator` region should start inside the guard.
        "expected_generator_start_min": 5000,
    },
    "b775cf29-8b42-4a83-af21-53075fef0802": {
        "check_name": "multiline_ocr_artifact_dropped",
        # Multi-line OCR artifacts must not appear in section_path.
        "forbidden_section_path_substrs": ["position", "\n"],
        "expected_primary_equip_type": "Gas Turbine",
        "expected_primary_esn": "297651",
    },
}

def _load_artifact_0819_ab(doc_id: str):
    path = os.path.join(OUTPUT_DIR, f"{doc_id}.json")
    if not os.path.exists(path):
        return None, path
    with open(path, "r") as artifact_file:
        return json.load(artifact_file), path

summary_0819_ab = {
    "output_dir": OUTPUT_DIR,
    "results": {},
    "missing_artifacts": [],
}

for doc_id, expectations in TARGET_DOCS_0819_AB.items():
    artifact, artifact_path = _load_artifact_0819_ab(doc_id)
    if artifact is None:
        summary_0819_ab["missing_artifacts"].append({"doc_id": doc_id, "path": artifact_path})
        continue

    meta = artifact.get("metadata") or {}
    regions = artifact.get("regions") or []

    doc_report = {
        "artifact_path": artifact_path,
        "check_name": expectations["check_name"],
        "region_count": len(regions),
        "primary_equip_type": meta.get("primary_equip_type"),
        "primary_esn": meta.get("primary_esn"),
        "failures": [],
    }

    if "expected_primary_equip_type" in expectations:
        want = expectations["expected_primary_equip_type"]
        got = str(meta.get("primary_equip_type") or "")
        if got.strip().lower() != want.strip().lower():
            doc_report["failures"].append({
                "type": "primary_equip_type_mismatch", "want": want, "got": got,
            })

    if "expected_primary_esn" in expectations:
        want = str(expectations["expected_primary_esn"]).strip()
        got = str(meta.get("primary_esn") or "").strip()
        if got != want:
            doc_report["failures"].append({
                "type": "primary_esn_mismatch", "want": want, "got": got,
            })

    if "min_expected_region_count" in expectations:
        want = int(expectations["min_expected_region_count"])
        if len(regions) < want:
            doc_report["failures"].append({
                "type": "region_count_too_low", "want_at_least": want, "got": len(regions),
            })

    if "expected_generator_start_min" in expectations:
        want_min = int(expectations["expected_generator_start_min"])
        gen_starts = []
        for region in regions:
            rm = (region or {}).get("metadata") or {}
            sp = [str(x) for x in (rm.get("section_path") or [])]
            if sp and sp[-1] == "3 Generator":
                gen_starts.append(region.get("start", 0))
        if gen_starts and min(gen_starts) < want_min:
            doc_report["failures"].append({
                "type": "generator_region_starts_inside_front_matter_guard",
                "min_start": min(gen_starts),
                "guard_min": want_min,
                "hint": "front-matter `3 Generator` (TOC line) leaked past the guard",
            })

    if "forbidden_section_path_substrs" in expectations:
        needles = [n.lower() for n in expectations["forbidden_section_path_substrs"]]
        offending = []
        for idx, region in enumerate(regions):
            rm = (region or {}).get("metadata") or {}
            sp = [str(x) for x in (rm.get("section_path") or [])]
            for elem in sp:
                lowered = elem.lower()
                for needle in needles:
                    if needle in lowered:
                        offending.append({
                            "region_index": idx,
                            "needle": needle,
                            "section_path": sp,
                        })
                        break
                else:
                    continue
                break
        if offending:
            doc_report["failures"].append({
                "type": "forbidden_section_path_substr_present",
                "count": len(offending),
                "sample": offending[:3],
            })

    doc_report["result"] = "PASS" if not doc_report["failures"] else "FAIL"
    summary_0819_ab["results"][doc_id] = doc_report

print("=== 8-19 OPTIONS A + B TARGET DOC SANITY ===")
print("output_dir:", summary_0819_ab["output_dir"])
print("missing_artifacts:", len(summary_0819_ab["missing_artifacts"]))
for missing in summary_0819_ab["missing_artifacts"]:
    print(" -", missing["doc_id"], "->", missing["path"])

any_fail = False
for doc_id, row in summary_0819_ab["results"].items():
    print(f"\n{doc_id} [{row['check_name']}] -> {row['result']}")
    print(" - region_count:", row["region_count"])
    print(" - primary_equip_type:", row["primary_equip_type"])
    print(" - primary_esn:", row["primary_esn"])
    if row["failures"]:
        any_fail = True
        for failure in row["failures"]:
            print(" - FAIL:", failure)

summary_0819_ab["overall_result"] = "PASS" if (
    not any_fail and not summary_0819_ab["missing_artifacts"]
) else "FAIL"
print("\noverall_result:", summary_0819_ab["overall_result"])
summary_0819_ab